# Step 1 – Exploring and downloading possible Biomass stacks over Antarctica

This code is adapted from the Examples_and_Data/Coding/Biomass/Data Access/Biomass Data Access Example.ipynb

### 1. Packages and Libraries

In [3]:
# %pip install aiohttp requests netCDF4 xarray folium shapely pillow

from shapely.geometry import Polygon
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os

import polsar_utils as pu
from BPS_inputs_download import get_stack
from pathlib import Path

fiona.supported_drivers["KML"] = "rw"  # Enable kml driver

### 2. Query the ESA MAAP STAC catalog

In [4]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

#### 2.1. Find the collection

In [5]:
# find all Biomass collections
results = catalog.collection_search(
    filter="platform='Biomass'",
    datetime="2025-12-01T00:00:00.000Z/..",
)
print(f"{results.matched()} collections found.")

data = results.collection_list_as_dict()
df = pd.json_normalize(data, record_path=["collections"]).sort_values(
    by="title", ignore_index=True
)
df[["id", "title", "extent.temporal.interval"]]

17 collections found.


,id,title,extent.temporal.interval
0,BiomassAux,Biomass Auxiliary,"[[2024-07-31T00:00:00.000Z, None]]"
1,BiomassAuxIOC,Biomass Auxiliary (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
2,BiomassAuxRest,Biomass Auxiliary Restricted,"[[2024-07-31T00:00:00.000Z, None]]"
3,BiomassCalVal10,Biomass Cal/Val,"[[2025-04-29T00:00:00.000Z, None]]"
4,BiomassLevel0,Biomass Level 0,"[[2024-07-31T00:00:00.000Z, None]]"
5,BiomassLevel0IOC,Biomass Level 0 (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
6,BiomassLevel1a,Biomass Level 1A,"[[2025-11-21T00:00:00.000Z, None]]"
7,BiomassLevel1aIOC,Biomass Level 1A (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
8,BiomassLevel1b,Biomass Level 1B,"[[2024-07-31T00:00:00.000Z, None]]"
9,BiomassLevel1bIOC,Biomass Level 1B (IOC),"[[2024-07-31T00:00:00.000Z, None]]"


#### 2.2. Select L1C products and narrow down the search

We are selecting L1C BIOMASS products as they contain stacks of scenes suitable for interferometry and tomography. The available querables are available here: [Queryables for L1c products](https://catalog.maap.eo.esa.int/catalogue/collections/BiomassLevel1c/queryables).

In [7]:
search = catalog.search(
    collections = ["BiomassLevel1c"],
    filter = "product:type='S1_STA__1S' or product:type='S2_STA__1S' or product:type='S3_STA__1S'",
    datetime = ['2025-01-01T00:00:00Z','2026-10-13T23:59:59Z'],
    # bbox = [-75.1451, -71.5351, -69.0196, -69.4491],  # Wilkins Ice Shelf, Antarctica
    # bbox = [94.6319, -67.9541, 103.7777, -63.7139], # Denman Glacier
    #bbox = [-107.7404, -75.5030, -98.5351, -73.3782], # PIG
    # bbox = [-157.0226, -84.6933, 163.2812, -75.9024], # Ross Ice shelf
    # bbox = [-126.0448, -57.8721, 128.0156, -53.8483], # whole Antarctica
    bbox = [7.272949,-71.566100,13.996582,-69.660815], #Nivilsen
    method = "GET",
    max_items = 320,  # change as needed
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

Accessing 229 items (limited by max_items).
229 items found that matched the query.


#### 2.3. Visualise the returned IDs into MAAP

In [8]:
features = []
for it in items:
    if it.geometry is None:
        raise ValueError(f"Item {it.id} has no geometry (footprint).")
    props = it.properties or {}
    features.append({
        "type": "Feature",
        "geometry": it.geometry,
        "properties": {
            "product_id": it.id,
            "stack_id": props.get("eofeos:stack_id"),  
            "start_datetime": props.get("start_datetime"),
            "end_datetime": props.get("end_datetime"),
        },
    })

gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
gdf = gdf.reset_index()
gdf['stack_number'] = gdf.groupby('stack_id').ngroup() + 1
gdf['products_in_stack'] = gdf.groupby('stack_id')['product_id'].transform('count')
gdf = gdf[["index", "product_id", "stack_id", "stack_number", "products_in_stack", "start_datetime", "end_datetime", "geometry"]]
gdf.head(len(items))

,index,product_id,stack_id,stack_number,products_in_stack,start_datetime,end_datetime,geometry
0,0,BIO_S1_STA__1S_20261004T060510_20261004T060538...,BIO_S1_STA__ID_20260916T060456_20261004T060538...,10,7,2026-10-04T06:05:10.151Z,2026-10-04T06:05:38.378Z,"POLYGON ((13.16671 -68.58579, 11.99278 -68.718..."
1,1,BIO_S1_STA__1S_20261004T060447_20261004T060511...,BIO_S1_STA__ID_20260916T060434_20261004T060511...,9,7,2026-10-04T06:04:47.727Z,2026-10-04T06:05:11.537Z,"POLYGON ((14.61241 -70.13889, 13.36837 -70.278..."
2,2,BIO_S1_STA__1S_20261002T161629_20261002T161651...,BIO_S1_STA__ID_20260914T161616_20261002T161651...,8,7,2026-10-02T16:16:29.422Z,2026-10-02T16:16:51.395Z,"POLYGON ((8.71478 -71.52021, 9.94272 -71.65664..."
3,3,BIO_S1_STA__1S_20261002T161610_20261002T161630...,BIO_S1_STA__ID_20260914T161556_20261002T161630...,7,7,2026-10-02T16:16:10.375Z,2026-10-02T16:16:30.810Z,"POLYGON ((9.92232 -70.33052, 11.07013 -70.4593..."
4,4,BIO_S1_STA__1S_20261002T063728_20261002T063753...,BIO_S1_STA__ID_20260914T063715_20261002T063753...,6,7,2026-10-02T06:37:28.732Z,2026-10-02T06:37:53.630Z,"POLYGON ((6.4352 -70.13908, 5.20496 -70.27671,..."
...,...,...,...,...,...,...,...,...
224,224,BIO_S2_STA__1S_20260514T160710_20260514T160730...,BIO_S2_STA__ID_20260514T160710_20260601T160741...,15,7,2026-05-14T16:07:10.304Z,2026-05-14T16:07:30.738Z,"POLYGON ((11.15754 -71.43904, 12.88048 -71.627..."
225,225,BIO_S2_STA__1S_20260514T160651_20260514T160711...,BIO_S2_STA__ID_20260514T160651_20260601T160722...,14,7,2026-05-14T16:06:51.257Z,2026-05-14T16:07:11.691Z,"POLYGON ((12.25503 -70.33636, 13.90245 -70.517..."
226,226,BIO_S2_STA__1S_20260514T062833_20260514T062853...,BIO_S2_STA__ID_20260514T062833_20260601T062904...,13,7,2026-05-14T06:28:33.169Z,2026-05-14T06:28:53.604Z,"POLYGON ((7.62851 -69.04302, 6.09406 -69.21351..."
227,227,BIO_S2_STA__1S_20260514T062814_20260514T062834...,BIO_S2_STA__ID_20260514T062814_20260601T062845...,12,7,2026-05-14T06:28:14.119Z,2026-05-14T06:28:34.553Z,"POLYGON ((8.68835 -70.14246, 7.07827 -70.32021..."


In [9]:
## Ensure both are WGS84 for folium
gdf = gdf.to_crs("EPSG:4326")

# Center/zoom using bounds
bounds = gpd.GeoSeries(list(gdf.geometry), crs="EPSG:4326")
minx, miny, maxx, maxy = bounds.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=3)

def style(_):
    return {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.10}  

GeoJson(
    gdf.__geo_interface__,
    name=f"Products ({len(gdf)})",
    tooltip=folium.GeoJsonTooltip(fields=["product_id", "index", "stack_id", "stack_number"]),
    style_function=style,
).add_to(m)


folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m

#### 2.4. Choose your stack

By hovering over the map above, you can see the product ID, the index in the geodatabase and the stack ID and stack number (stack number is a unique number for each stack_id, as stack_id is a very long string).

Pick the stack you want to download and insert the stack_number in the line below.

In [10]:
CHOSEN_STACK_NUMBER = 4 #pick stack number

In [11]:
gdf_subset = gdf[gdf['stack_number'] == CHOSEN_STACK_NUMBER].copy()

# Ensure both are WGS84 for folium
gdf_subset = gdf_subset.to_crs("EPSG:4326")

# Center/zoom using bounds of the filtered stack
bounds = gpd.GeoSeries(list(gdf_subset.geometry), crs="EPSG:4326")
minx, miny, maxx, maxy = bounds.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=6)

def style(_):
    return {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.20}  

GeoJson(
    gdf_subset.__geo_interface__,
    name=f"Stack {CHOSEN_STACK_NUMBER} ({len(gdf_subset)} products)",
    tooltip=folium.GeoJsonTooltip(
        fields=["product_id", "index", "stack_id", "stack_number", "products_in_stack"],
        aliases=["Product ID:", "Index:", "Stack ID:", "Stack Number:", "Total in Stack:"]
    ),
    style_function=style,
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m

#### 3.4. Download your stack

Download the stack to the chosen DOWNLOAD_FOLDER.

In [12]:
DOWNLOAD_FOLDER = Path("Input_biomass")

In [13]:
from collections import defaultdict
stacks = defaultdict(list)
for idx, item in enumerate(items):
    stack_id = (item.properties or {}).get("eofeos:stack_id")
    if stack_id:
        stacks[stack_id].append(idx)
 
# Use your chosen stack ID string directly (e.g., from your gdf subset)
chosen_stack_id = gdf[gdf['stack_number'] == CHOSEN_STACK_NUMBER]['stack_id'].iloc[0]
if chosen_stack_id not in stacks:
    raise ValueError(f"Stack ID {chosen_stack_id} not found in the items list.")
 
indices = stacks[chosen_stack_id]
 
reference_id = items[indices[0]].id
 
safe_stack_name = chosen_stack_id.replace(":", "_").replace("/", "_")
stack_folder = DOWNLOAD_FOLDER / safe_stack_name
stack_folder.mkdir(parents=True, exist_ok=True)

In [14]:
print(f"\nDownloading Stack {CHOSEN_STACK_NUMBER} | Stack ID: {stack_id}")
print(f"Catalog indices in stack: {indices}")
print(f"Reference L1C ID: {reference_id}")
print(f"Saving to: {stack_folder.resolve()}")

# Perform the download
get_stack(reference_id, str(stack_folder))
print(f"Completed downloading stack {stack_id}.")


Catalog indices in stack: [3, 8, 13, 18, 23, 28, 33]
Reference L1C ID: BIO_S1_STA__1S_20261002T161610_20261002T161630_T_G01_M05_C07_T010_F216_02_DYXAGJ
Saving to: /home/jovyan/idas_dump/test/Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS
Matching parameters for BIO_S1_STA__1S_20261002T161610_20261002T161630_T_G01_M05_C07_T010_F216_02_DYXAGJ: G01 M05 T010 F216 baseline=02
Products to be downloaded in Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS (7):
- BIO_S1_STA__1S_20261002T161610_20261002T161630_T_G01_M05_C07_T010_F216_02_DYXAGJ
- BIO_S1_STA__1S_20260929T161608_20260929T161628_T_G01_M05_C06_T010_F216_02_DYXAFW
- BIO_S1_STA__1S_20260926T161605_20260926T161626_T_G01_M05_C05_T010_F216_02_DYXAFA
- BIO_S1_STA__1S_20260923T161603_20260923T161624_T_G01_M05_C04_T010_F216_02_DYXAEM
- BIO_S1_STA__1S_20260920T161601_20260920T161621_T_G01_M05_C03_T010_F216_02_DYXAE3
- BIO_S1_STA__1S_20260917T161559_

BIO_S1_STA__1S_20261002T161610_20261002T161630_T_G01_M05_C07_T010_F216_02_DYXAGJ.zip: 338MiB [00:39, 8.98MiB/s] 


Unzipping BIO_S1_STA__1S_20261002T161610_20261002T161630_T_G01_M05_C07_T010_F216_02_DYXAGJ into Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/L1C
Removing BIO_S1_STA__1S_20261002T161610_20261002T161630_T_G01_M05_C07_T010_F216_02_DYXAGJ.zip to save memory
Deleted Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/BIO_S1_STA__1S_20261002T161610_20261002T161630_T_G01_M05_C07_T010_F216_02_DYXAGJ.zip


BIO_S1_STA__1S_20260929T161608_20260929T161628_T_G01_M05_C06_T010_F216_02_DYXAFW.zip: 339MiB [00:26, 13.6MiB/s] 


Unzipping BIO_S1_STA__1S_20260929T161608_20260929T161628_T_G01_M05_C06_T010_F216_02_DYXAFW into Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/L1C
Removing BIO_S1_STA__1S_20260929T161608_20260929T161628_T_G01_M05_C06_T010_F216_02_DYXAFW.zip to save memory
Deleted Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/BIO_S1_STA__1S_20260929T161608_20260929T161628_T_G01_M05_C06_T010_F216_02_DYXAFW.zip


...

Unzipping BIO_S1_STA__1S_20260920T161601_20260920T161621_T_G01_M05_C03_T010_F216_02_DYXAE3 into Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/L1C
Removing BIO_S1_STA__1S_20260920T161601_20260920T161621_T_G01_M05_C03_T010_F216_02_DYXAE3.zip to save memory
Deleted Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/BIO_S1_STA__1S_20260920T161601_20260920T161621_T_G01_M05_C03_T010_F216_02_DYXAE3.zip


BIO_S1_STA__1S_20260917T161559_20260917T161619_T_G01_M05_C02_T010_F216_02_DYXADH.zip: 339MiB [00:27, 12.8MiB/s] 


Unzipping BIO_S1_STA__1S_20260917T161559_20260917T161619_T_G01_M05_C02_T010_F216_02_DYXADH into Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/L1C
Removing BIO_S1_STA__1S_20260917T161559_20260917T161619_T_G01_M05_C02_T010_F216_02_DYXADH.zip to save memory
Deleted Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/BIO_S1_STA__1S_20260917T161559_20260917T161619_T_G01_M05_C02_T010_F216_02_DYXADH.zip


BIO_S1_STA__1S_20260914T161556_20260914T161617_T_G01_M05_C01_T010_F216_02_DYXACU.zip: 338MiB [00:26, 13.5MiB/s] 


Unzipping BIO_S1_STA__1S_20260914T161556_20260914T161617_T_G01_M05_C01_T010_F216_02_DYXACU into Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/L1C
Removing BIO_S1_STA__1S_20260914T161556_20260914T161617_T_G01_M05_C01_T010_F216_02_DYXACU.zip to save memory
Deleted Input_biomass/BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS/BIO_S1_STA__1S_20260914T161556_20260914T161617_T_G01_M05_C01_T010_F216_02_DYXACU.zip
Completed downloading stack BIO_S1_STA__ID_20260914T161556_20261002T161630_T_G01_M05_S0698_E0111_D_02_DYX7JS.


#### 3.5. Inspect

In [16]:
stacks = pu.find_stacks(DOWNLOAD_FOLDER)
pu.print_overview(stacks)

1 stack(s), 7 scene(s)

[0] BIO_S1_STA__ID_20260915T214352_20261003T214425_T_G01_M05_S0720_W0730_D_02_DZ1NQJ  (7 scenes)
      [0] 2026-09-15 21:43            BIO_S1_STA__1S_20260915T214352_20260915T214412_T_G01_M05_C01_T028_F218_02_DZ1QL2
      [1] 2026-09-18 21:43            BIO_S1_STA__1S_20260918T214354_20260918T214414_T_G01_M05_C02_T028_F218_02_DZ1QLP
      [2] 2026-09-21 21:43            BIO_S1_STA__1S_20260921T214356_20260921T214417_T_G01_M05_C03_T028_F218_02_DZ1QMC
      [3] 2026-09-24 21:43  ★ master  BIO_S1_STA__1S_20260924T214358_20260924T214419_T_G01_M05_C04_T028_F218_02_DZ1QMU
      [4] 2026-09-27 21:44            BIO_S1_STA__1S_20260927T214401_20260927T214421_T_G01_M05_C05_T028_F218_02_DZ1QNE
      [5] 2026-09-30 21:44            BIO_S1_STA__1S_20260930T214403_20260930T214423_T_G01_M05_C06_T028_F218_02_DZ1QO2
      [6] 2026-10-03 21:44            BIO_S1_STA__1S_20261003T214405_20261003T214425_T_G01_M05_C07_T028_F218_02_DZ1QOO

★ = master (coregistration primary) of the st